In [ ]:
"""Blend file41 (6.576 real LB, older 0.594-lineage base + beam2/neighbor/WARP) with file44
(6.464 real LB, newer 0.462-lineage base + beam2/neighbor/WARP+Q0522 hedge). NOTE: unlike the
p21+p44 blend, these two are NOT independent pipelines -- diff check showed they agree exactly
on 2/3 real test wells and differ only on 00e12e8b (mean 0.35ft), which is precisely the well the
competitor's Q0522 cell hand-tunes against the public score. This blend is really a half-dose of
that specific hedge, not a cross-lineage ensemble -- testing whether the hedge is LB-overfit.
v4: after two straight submission-scoring exceptions on brand-new private datasets (single
combined, then split-into-two), switched to referencing the two SOURCE KERNELS directly via
kernel_sources instead of packaging a new dataset -- kernels 41 and 44 have already been through
many real successful submissions, so their outputs should already be 'warm' in whatever mount
path Kaggle's scoring environment uses, unlike a dataset created minutes/hours ago.
"""
import glob, os, time, pandas as pd

BLEND_W = 0.50  # weight on p44; (1-BLEND_W) on p41

def _wait_for(pattern, tries=10, delay=15):
    for i in range(tries):
        hits = glob.glob(pattern, recursive=True)
        if hits:
            if i > 0:
                print(f'{pattern}: found after {i} retries', flush=True)
            return hits
        print(f'{pattern}: not found yet (try {i+1}/{tries}), waiting {delay}s', flush=True)
        time.sleep(delay)
    return []

_all_subs = _wait_for('/kaggle/input/**/submission.csv')
assert _all_subs, 'no submission.csv found -- attach kernel_sources 41-fork... and rogii-file44-v2'
print('found submission.csv files:', _all_subs, flush=True)

_p41_path = next((p for p in _all_subs if '41-fork' in p or 'beam2-neighbor-warp' in p), None)
_p44_path = next((p for p in _all_subs if 'file44' in p or '-v2' in p), None)
assert _p41_path and _p44_path and _p41_path != _p44_path, (
    f'could not disambiguate p41/p44 among {_all_subs}')

_comp = _wait_for('/kaggle/input/**/sample_submission.csv')
assert _comp, 'sample_submission.csv not found -- attach the competition dataset'
SAMPLE = _comp[0]

p41 = pd.read_csv(_p41_path, dtype={'id': 'string'})
p44 = pd.read_csv(_p44_path, dtype={'id': 'string'})
sample = pd.read_csv(SAMPLE, dtype={'id': 'string'})[['id']]

p41 = sample.merge(p41, on='id', how='left')
p44 = sample.merge(p44, on='id', how='left')
assert p41['tvt'].notna().all() and p44['tvt'].notna().all(), 'missing ids after merge'
assert (p41['id'] == p44['id']).all() and (p41['id'] == sample['id']).all()

blended = sample.copy()
blended['tvt'] = (1.0 - BLEND_W) * p41['tvt'].to_numpy(float) + BLEND_W * p44['tvt'].to_numpy(float)

assert len(blended) == len(sample)
assert blended['tvt'].notna().all()
blended[['id', 'tvt']].to_csv('submission.csv', index=False)
print(f'blend_w(p44)={BLEND_W}  rows={len(blended)}  '
      f'tvt mean={blended["tvt"].mean():.3f} std={blended["tvt"].std():.3f}')
print('saved submission.csv')